# 08 - Visualize Predictions (presentation artifacts)

Shows how the trained model behaves, for slides/demos. Produces, under
`reports/<run_id>/viz/`:

1. **Per-class example grids** - N test frames per class with GT (green) vs
   prediction (red + confidence), plus the deployment category
   (Low / High / Low Unknown / High Unknown).
2. **A "hero" contact sheet** - one clean, correct example per class in one image.
3. **Annotated approach videos** - full mid-approach clips per class with live
   boxes drawn, saved as MP4 for slides.
4. **A low/high confusion figure** - the USS-assist headline.

Reads the trained model, the dataset manifest (for test-split membership and
GT), and the raw sequence videos (for full clips). Materializes only the frames
it draws, so it is fast and needs no full local dataset.

## 1. Setup + resolve the model / dataset

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")

In [ ]:
import json as _json
import random
from collections import defaultdict

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from tqdm.auto import tqdm
from ultralytics import YOLO

from vision_uss_research.runs import new_run_id, resolve_run, start_run, finish_run
from vision_uss_research.sequences import (extract_sequence_frames, frames_window_id,
                                           select_camera_videos, with_drive_retry)

MODEL_RUN = None          # models/<run_id>; None = newest
CONF = 0.25
N_PER_CLASS = 6           # example frames per class in the grids
VIDEO_CLIPS_PER_CLASS = 1 # annotated approach videos to render per class
CLIP_FPS = 6
SEED = 0

# deployment category mapping (matches notebook 07)
CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "low_other": "low", "high_other": "high"}
CATEGORY_NAMES = {"low_other": "Low Unknown", "high_other": "High Unknown"}
def category_of(name):
    return CATEGORY_NAMES.get(name, CLASS_BINS.get(name, "?").capitalize())

model_dir = resolve_run(ARTIFACTS / "models", "train", run_id=MODEL_RUN,
                        require_file="train/weights/best.pt")
assert model_dir is not None, "no trained model - run notebook 06 first"
model_run = _json.loads((model_dir / "run.json").read_text())
dataset_dir = ARTIFACTS / Path(model_run["inputs"][0])
manifest_csv = dataset_dir / "dataset_manifest.csv"
assert manifest_csv.exists(), f"expected a manifest dataset at {dataset_dir}"
prov = _json.loads((dataset_dir / "provenance.json").read_text())
class_names = prov["class_names"]
frames_roots = {k: Path(v) for k, v in prov["frames_roots"].items()}

model = YOLO(str(model_dir / "train" / "weights" / "best.pt"))

VIZ_DIR = ARTIFACTS / "reports" / new_run_id("viz", "gallery") / "viz"
VIZ_DIR.mkdir(parents=True, exist_ok=True)
start_run(VIZ_DIR.parent, config={"model_run": model_dir.name, "conf": CONF},
          inputs=[f"models/{model_dir.name}", f"datasets/{dataset_dir.name}"],
          repo_root=REPO_DIR)
print(f"model   : {model_dir.name}")
print(f"dataset : {dataset_dir.name}")
print(f"classes : {class_names}")
print(f"output  : {VIZ_DIR}")

## 2. Test-split frames per class (from the manifest)

In [ ]:
from vision_uss_research.labeling.boxes_io import resolve_frame_path

manifest = pd.read_csv(manifest_csv)
test = manifest[manifest["split"] == "test"].copy()
test["class_name"] = test["class_id"].map(dict(enumerate(class_names)))

# one row per (sequence, camera, frame): keep its boxes grouped
frame_groups = {k: g for k, g in
                test.groupby(["class_name", "sequence_id", "camera", "frame_idx", "source"])}
by_class = defaultdict(list)
for (cname, seq, cam, fidx, src), g in frame_groups.items():
    by_class[cname].append((seq, cam, int(fidx), src, g))

print("test frames per class:")
for c in class_names:
    print(f"  {c:<14} {len(by_class.get(c, []))} frames")

## 3. Drawing helpers

GT in green, predictions in red with confidence and the deployment category.

In [ ]:
COLOR_GT = (40, 200, 60)
COLOR_PRED = (235, 60, 60)

def load_frame(seq, cam, fidx, src):
    row = {"sequence_id": seq, "camera": cam, "source": src,
           "frame_path": f"{seq}/{cam}/{fidx:05d}.jpg"}
    p = resolve_frame_path(row, frames_roots[str(src)])
    if p is None:
        return None
    img = cv2.imread(str(p))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else None

def draw_gt(img, grp):
    for _, r in grp.iterrows():
        cv2.rectangle(img, (int(r.x0), int(r.y0)), (int(r.x1), int(r.y1)), COLOR_GT, 2)
    return img

def draw_preds(img, result):
    h = img.shape[0]
    for cls, box, conf in zip(result.boxes.cls.tolist(),
                              result.boxes.xyxy.tolist(),
                              result.boxes.conf.tolist()):
        x0, y0, x1, y1 = map(int, box)
        name = class_names[int(cls)]
        cv2.rectangle(img, (x0, y0), (x1, y1), COLOR_PRED, 2)
        label = f"{category_of(name)}:{name} {conf:.2f}"
        cv2.putText(img, label, (x0, min(y1 + 20, h - 6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_PRED, 2)
    return img

def predict_draw(seq, cam, fidx, src, grp=None):
    img = load_frame(seq, cam, fidx, src)
    if img is None:
        return None
    canvas = img.copy()
    if grp is not None:
        draw_gt(canvas, grp)
    draw_preds(canvas, model.predict(img[..., ::-1], conf=CONF, verbose=False)[0])
    return canvas

## 4. Per-class example grids

In [ ]:
rng = random.Random(SEED)

def grid(images, titles, title, cols=3, save=None):
    n = len(images)
    if n == 0:
        print(f"{title}: no frames"); return
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, im, t in zip(axes.flat, images, titles):
        ax.imshow(im); ax.set_title(t, fontsize=9)
    fig.suptitle(title, fontsize=14)
    if save:
        fig.savefig(save, dpi=95, bbox_inches="tight")
    plt.show()

for cname in class_names:
    frames = by_class.get(cname, [])
    if not frames:
        continue
    picks = rng.sample(frames, min(N_PER_CLASS, len(frames)))
    imgs, titles = [], []
    for seq, cam, fidx, src, grp in picks:
        canvas = predict_draw(seq, cam, fidx, src, grp)
        if canvas is not None:
            imgs.append(canvas)
            titles.append(f"{seq[:22]} ({cam} f{fidx})")
    grid(imgs, titles, f"{cname}  -  {category_of(cname)}   (GT green / pred red)",
         save=VIZ_DIR / f"grid_{cname}.jpg")

## 5. Hero contact sheet (one clean correct example per class)

In [ ]:
def best_correct_example(cname):
    """A frame where the model predicts cname with high confidence and the GT
    box is present - the cleanest slide example."""
    best = None
    for seq, cam, fidx, src, grp in by_class.get(cname, []):
        img = load_frame(seq, cam, fidx, src)
        if img is None:
            continue
        res = model.predict(img[..., ::-1], conf=CONF, verbose=False)[0]
        confs = [c for cl, c in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist())
                 if class_names[int(cl)] == cname]
        if confs and (best is None or max(confs) > best[0]):
            canvas = img.copy(); draw_gt(canvas, grp); draw_preds(canvas, res)
            best = (max(confs), canvas, f"{cname} ({category_of(cname)})")
        if best and best[0] > 0.85:
            break
    return best

heroes = [h for h in (best_correct_example(c) for c in class_names) if h]
grid([h[1] for h in heroes], [h[2] for h in heroes],
     "Model at a glance - one example per class", cols=3,
     save=VIZ_DIR / "hero_contact_sheet.jpg")

## 6. Annotated approach videos

Full mid-approach clip per class with live prediction boxes - the most
convincing demo artifact. Extracts the frame window from the raw video (shared
cache), draws predictions on every frame, writes an MP4.

In [ ]:
N_FRAMES = 20
RATIO_RANGE = (0.30, 0.85)
FRAMES_ROOT = ARTIFACTS / "frames"

WINDOW_ROOT = FRAMES_ROOT / frames_window_id(N_FRAMES, RATIO_RANGE)

def render_clip(cname, seq, cam, src, out_path):
    # frames were cached during labeling; read them straight from the cache
    frames_dir = WINDOW_ROOT / seq / cam
    imgs = sorted(frames_dir.glob("*.jpg")) if frames_dir.exists() else []
    if not imgs:
        return False
    first = cv2.imread(str(imgs[0]))
    h, w = first.shape[:2]
    writer = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*"mp4v"),
                             CLIP_FPS, (w, h))
    for ip in imgs:
        rgb = cv2.cvtColor(cv2.imread(str(ip)), cv2.COLOR_BGR2RGB)
        draw_preds(rgb, model.predict(rgb[..., ::-1], conf=CONF, verbose=False)[0])
        cv2.putText(rgb, f"{cname} -> {category_of(cname)}", (12, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 0), 2)
        writer.write(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR))
    writer.release()
    return True

clip_dir = VIZ_DIR / "clips"
clip_dir.mkdir(exist_ok=True)
rendered = []
for cname in class_names:
    picks = rng.sample(by_class.get(cname, []), min(VIDEO_CLIPS_PER_CLASS,
                                                    len(by_class.get(cname, []))))
    for seq, cam, fidx, src, grp in picks:
        out = clip_dir / f"{cname}_{seq}_{cam}.mp4"
        if render_clip(cname, seq, cam, src, out):
            rendered.append(out)
            print(f"clip: {out.name}")
print(f"\n{len(rendered)} clips -> {clip_dir}")

In [ ]:
# preview one clip inline (Colab)
if IN_COLAB and rendered:
    from IPython.display import HTML
    from base64 import b64encode
    data = b64encode(rendered[0].read_bytes()).decode()
    display(HTML(f'<video width=520 controls><source src="data:video/mp4;base64,{data}"></video>'))

## 7. Low/High confusion figure (USS-assist headline)

In [ ]:
# reuse the eval report's bin confusion if present, else compute a quick one
eval_dir = resolve_run(ARTIFACTS / "reports", "eval", require_file="run.json")
conf_mat = None
if eval_dir is not None:
    summ = _json.loads((eval_dir / "run.json").read_text()).get("summary", {})
    bc = summ.get("bin_confusion")
    if bc:
        conf_mat = pd.DataFrame(0, index=["low", "high"], columns=["low", "high"])
        for k, n in bc.items():
            g, p = k.split("->")
            conf_mat.loc[g, p] = n

if conf_mat is not None:
    fig, ax = plt.subplots(figsize=(4.6, 4))
    im = ax.imshow(conf_mat.values, cmap="Blues")
    ax.set_xticks([0, 1], ["pred low", "pred high"])
    ax.set_yticks([0, 1], ["true low", "true high"])
    for i in range(2):
        for j in range(2):
            v = conf_mat.values[i, j]
            ax.text(j, i, str(v), ha="center", va="center",
                    color="white" if v > conf_mat.values.max() / 2 else "black",
                    fontsize=14)
    acc = np.trace(conf_mat.values) / max(conf_mat.values.sum(), 1)
    ax.set_title(f"Low/High on matched detections\naccuracy = {acc:.3f}")
    fig.savefig(VIZ_DIR / "lowhigh_confusion.jpg", dpi=110, bbox_inches="tight")
    plt.show()
else:
    print("no eval report found - run notebook 07 first for the confusion figure")

## 8. Save + index

In [ ]:
artifacts = sorted(p.name for p in VIZ_DIR.rglob("*") if p.is_file())
finish_run(VIZ_DIR.parent, summary={"n_artifacts": len(artifacts),
                                    "clips": len(rendered)})
print(f"presentation artifacts in {VIZ_DIR}:")
for a in artifacts:
    print(f"  {a}")

## Notes for slides

- **hero_contact_sheet.jpg** - the single "what it does" figure.
- **grid_<class>.jpg** - per-class deep dive; pair a strong class (speedbump,
  dummychild) with the honest weak one (high_other / held-out cone) to tell the
  open-set story truthfully.
- **clips/*.mp4** - drop straight into slides; the live boxes over an approach
  are the most convincing artifact.
- **lowhigh_confusion.jpg** - the USS-assist headline (near-perfect Low/High).
